In [ ]:
using WAVI, Plots, Downloads, LinearAlgebra, MAT, JLD2

BLAS.set_num_threads(1)

In [ ]:
# matlab

#
#Grid and boundary conditions
# 
# nx = 105
# ny = 123
# nσ = 12
# x0 = -1812000.
# y0 = -860000.
# dx = 8000.0
# dy = 8000.0

# inv_in = "/home/tgia02/repos/WAVI.jl/example_drivers/Rosie_examples/bslcenb_data_icesheet_output_chll1/wavi_setups_Mira/wavi_8km_2015_example/"
# inv_out = "/home/tgia02/repos/WAVI.jl/example_drivers/Rosie_examples/driver_wavi_8km_2015_example_Mira_output/"

# constructor

nx = 106#162 # use the clipped grid
ny = 123#177 # use the clipped grid
nσ = 26 # tried with less than 26, but could not reach sigma=1.0 and got error. seems to be given by contructor but cannot identify where exactly
x0 = -1.812e6#-2260000.0
y0 = -860000.0#-1292000.0
dx = 8000.0
dy = 8000.0

inv_in = "/home/tgia02/repos/WAVIConstructor.jl/examples/outputs/amundsen_verification/"
inv_out = "/home/tgia02/repos/WAVI.jl/example_drivers/Rosie_examples/driver_wavi_8km_Apr_2026_Amundsen_verification_output/"


In [ ]:
inv_out_data = matread(inv_out*"outfile0000000025.mat")

In [ ]:
f = inv_out_data["preBfactor"]
#heatmap(log.(f)')
heatmap(f')

In [ ]:
f = inv_out_data["av_speed"]
heatmap(log.(f)')

In [ ]:
inv_out_checkpoint = load(inv_out*"PChkpt_0000000025.jld2")

In [ ]:
initial_temp_checkpoint = inv_out_checkpoint["inversion_simulation"].model.initial_conditions.initial_temperature;
heatmap(initial_temp_checkpoint[:,:,1]')

In [ ]:
initial_temp_inv_in=Array{Float64}(undef,nx,ny,nσ);
read!(inv_in*"/temps.bin",initial_temp_inv_in)
initial_temp_inv_in.=ntoh.(initial_temp_inv_in);

heatmap(initial_temp_inv_in[:,:,1]')

In [ ]:
fields_3d = inv_out_checkpoint["inversion_simulation"].model.fields.g3d

In [ ]:
f = fields_3d.θ
#heatmap(log.(f)[:,:,1]')
heatmap(f[:,:,1]')

In [ ]:
f = fields_3d.Φ
println(minimum(f))
println(maximum(f))
println(sum(f))
heatmap(f[:,:,1]')


In [ ]:
# We need these before we can build a grid
h_mask=Array{Float64}(undef,nx,ny);
read!(inv_in*"/h_mask.bin",h_mask)
#h_mask.=ntoh.(h_mask)

u_iszero=Array{Float64}(undef,nx+1,ny);
read!(inv_in*"/u_iszero.bin",u_iszero)
#u_iszero.=ntoh.(u_iszero)

v_iszero=Array{Float64}(undef,nx,ny+1);
read!(inv_in*"/v_iszero.bin",v_iszero)
#v_iszero.=ntoh.(v_iszero)

sigma_grid=Array{Float64}(undef,nσ);
read!(inv_in*"/sigma_grid.bin",sigma_grid)
#sigma_grid.=ntoh.(sigma_grid)

basin_ID=Array{Float64}(undef,nx,ny);
read!(inv_in*"/basinID.bin",basin_ID)
#basin_ID.=ntoh.(basin_ID)

grid = Grid(nx = nx, 
            ny = ny,   
            nσ = nσ, 
            x0 = x0, 
            y0 = y0, 
            dx = dx, 
            dy = dy,
            h_mask = h_mask, 
            u_iszero = u_iszero, 
            v_iszero = v_iszero,
            σ = sigma_grid,
            basin_ID = basin_ID)

In [ ]:
#
#Bed 
#
bed=Array{Float64}(undef,nx,ny);
read!(inv_in*"/bed.bin",bed)
#bed.=ntoh.(bed)

# Let's take a look at the bed
plt = Plots.heatmap(grid.xxh[:,1]/1e3, grid.yyh[1,:]/1e3, bed', 
                    xlabel = "x (km)", 
                    ylabel = "y (km)",
                    colorbar_title = "bed elevation (m)",
                    title = "bed elevation",
                    framestyle = "box")

In [ ]:
# Temperature, damage, viscosity, and thickness will be passed to a model via initial conditions (these quantities are time dependent)
temp = fields_3d.θ
damage = fields_3d.Φ
viscosity = fields_3d.η

# thickness
h = inv_out_data["h"]

heatmap(h')

In [ ]:
solver_params=SolverParams(maxiter_picard=20)

initial_conditions = InitialConditions(initial_thickness = h,initial_viscosity = viscosity,initial_temperature = temp,initial_damage = damage);

parallel_spec = BasicParallelSpec()

# ## Ice Velocity
# Now we're ready to make our model, which we can then use to determine the ice velocity. All physical and solver parameters take their default values
serial_model = Model(grid = grid, bed_elevation = bed,initial_conditions= initial_conditions, parallel_spec = parallel_spec, solver_params=solver_params);

# We use the `update_state!` method to bring fields (including velocity) in line with the ice thickness:
update_state!(serial_model)

In [ ]:
# Now we can visualize the ice velocity:plt = Plots.heatmap(grid.xxh[:,1]/1e3, grid.yyh[1,:]/1e3, log10.(serial_model.fields.gh.av_speed'), 
plt = Plots.heatmap(grid.xxh[:,1]/1e3, grid.yyh[1,:]/1e3, log10.(serial_model.fields.gh.av_speed'), 
                    xlabel = "x (km)", 
                    ylabel = "y (km)",
                    colorbar_title = "ice speed (m/yr)",
                    title = "Ice speed",
             #       clims=(-0.02,0.02),
             #       xlims=(-1400,-1300),
             #       ylims=(-500,-400),
                    framestyle = "box")

In [ ]:
# Now try the same velocity solve with a shared memory parallel implementation                     
saved_serial_model=deepcopy(serial_model);

parallel_model = deepcopy(serial_model)
parallel_model = @set parallel_model.solver_params = SolverParams(maxiter_picard=5)
parallel_model = @set parallel_model.parallel_spec = SharedMemorySpec(ngridsx=2,ngridsy=3,overlap=1,damping=0.0,niterations=1)

cols=cgrad(:roma,rev=true);
nSchwarzIterations=1
for iSchwarzIteration = 1:nSchwarzIterations
       update_state!(parallel_model)


       # Now we can visualize the ice velocity:
       plt = Plots.heatmap(grid.xxh[:,1]/1e3, grid.yyh[1,:]/1e3, log10.(parallel_model.fields.gh.av_speed'), 
                     c = cols, 
                     xlabel = "x (km)", 
                     ylabel = "y (km)",
                     colorbar_title = "ice speed (m/yr)",
                     title = "Ice speed",
              #       clims=(-0.2,0.2),
              #       xlims=(-1600,-1500),
              #       ylims=(-700,-600),
                     framestyle = "box")
       display(plt)

       plt = Plots.heatmap(grid.xxh[:,1]/1e3, grid.yyh[1,:]/1e3, log10.(parallel_model.fields.gh.ηav'), 
       c = cols,
       xlabel = "x (km)", 
       ylabel = "y (km)",
       colorbar_title = "ice speed (m/yr)",
       title = "Ice speed",
#       clims=(-0.2,0.2),
#       xlims=(-1600,-1500),
#       ylims=(-700,-600),
       framestyle = "box")
       display(plt)

       plt = Plots.heatmap(grid.xxh[:,1]/1e3, grid.yyh[1,:]/1e3, log10.(parallel_model.fields.gh.av_speed'./serial_model.fields.gh.av_speed'), 
       c = cols,
       xlabel = "x (km)", 
       ylabel = "y (km)",
       colorbar_title = "ice speed (m/yr)",
       title = "Ice speed",
#       clims=(-0.2,0.2),
#       xlims=(-1600,-1500),
#       ylims=(-700,-600),
       framestyle = "box")
       display(plt)

# Ice velocities reach a maximum of approx 5km/yr on ice shelves, but are much smaller inland.

       plt = Plots.heatmap(grid.xxu[:,1]/1e3, grid.yyu[1,:]/1e3, log10.(abs.(parallel_model.fields.gu.residual')), 
                     c = cols,
                     xlabel = "x (km)", 
                     ylabel = "y (km)",
                     colorbar_title = "ice speed (m/yr)",
                     title = "Ice speed",
              #       clims=(-0.02,0.02),
              #       xlims=(-1400,-1300),
              #       ylims=(-500,-400),
                     framestyle = "box")
       display(plt)

       plt = Plots.heatmap(grid.xxv[:,1]/1e3, grid.yyv[1,:]/1e3, log10.(abs.(parallel_model.fields.gv.residual')), 
                     c=cols,
                     xlabel = "x (km)", 
                     ylabel = "y (km)",
                     colorbar_title = "ice speed (m/yr)",
                     title = "Ice speed",
              #       clims=(-0.02,0.02),
              #       xlims=(-1400,-1300),
              #       ylims=(-500,-400),
                     framestyle = "box")
       display(plt)

 end

In [ ]:
Φ = parallel_model.fields.g3d.Φ
heatmap(grid.xxh[:,1]/1e3, grid.yyh[1,:]/1e3,
Φ[:,:,1]')

In [ ]:
xx= parallel_model.grid.xxh/1000
yy = parallel_model.grid.yyh/1000
s = parallel_model.fields.gh.s
u = parallel_model.fields.gh.u
v = parallel_model.fields.gh.v
Φ = parallel_model.fields.g3d.Φ

hm = inv_out_data["h_mask"]


Φ1 = copy(Φ[:,:,1])
Φ1[.!hm] .= NaN

clim = (0,1.0)
#cmap = colormap("Blues", logscale = true)
p1=heatmap(xx[:,1],yy[1,:],(Φ1[:,:,1]./hm)',clim = clim,
           xlabel="x (km)",ylabel="y (km)", 
           #colormap = cmap, 
           framestyle = :box)
